# SPARC rotation curve → CSV

Downloads a galaxy rotation curve from the [SPARC database](https://astroweb.cwru.edu/SPARC/) (Spitzer Photometry and Accurate Rotation Curves, Lelli et al. 2016) and writes a CSV file ready for analysis in Excel, Python, etc.

Steps:

1. Download the SPARC `_rotmod.dat` file for the chosen galaxy from the Rotmod_LTG.zip archive (requires internet access; raises an error if the download fails).
2. Scale the disk and bulge contributions by the mass-to-light ratios `ML_DISK` / `ML_BUL` (3.6 μm, solar units).
3. Combine the baryonic components: `Vbaryon = sqrt(Vgas² + ML_disk·Vdisk² + ML_bul·Vbul²)`.
4. Compute enclosed masses from `M[1e9 Msun] = V²[km/s]² · R[kpc] / 4302` (G = 4.302e-3 pc M☉⁻¹ (km/s)²), and the dark-matter fraction `fDM = (Mobs − Mbar) / Mobs`.
5. CSV output with columns:
   - `R_kpc` – galactocentric radius (kpc)
   - `Vobs_kms`, `errV_kms` – observed rotation velocity and its 1σ error (km/s)
   - `Vgas_kms`, `Vdisk_kms`, `Vbul_kms` – gas, disk and bulge contributions (km/s)
   - `Vbaryon_kms` – total baryonic velocity (km/s)
   - `Mobs_1e9Msun`, `Mbar_1e9Msun`, `MDM_1e9Msun` – dynamical, baryonic and dark-matter mass enclosed within R (10⁹ M☉)
   - `fDM` – dark-matter mass fraction

**Usage:** choose a galaxy in the settings cell and run all cells (Runtime → Run all). Leave `ML_DISK` / `ML_BUL` at `-1` to use the catalogue default for that galaxy. The CSV is downloaded automatically at the end.

Data: [SPARC](https://astroweb.cwru.edu/SPARC/) database. Reference: Lelli, McGaugh, Schombert, AJ 152, 157 (2016).

In [ ]:
GALAXY = "NGC3198"  #@param ["NGC3198", "NGC6503", "DDO154", "NGC2403"]
ML_DISK = -1.0      #@param {type:"number"}
ML_BUL = -1.0       #@param {type:"number"}
OUTPUT = ""         #@param {type:"string"}
# ML_DISK / ML_BUL: mass-to-light ratio of disk / bulge (3.6 micron, solar units).
#   Leave at -1 to use the catalogue default for the chosen galaxy.
# Leave OUTPUT empty to use <GALAXY>.csv

In [ ]:
import csv
import math
import sys
import io
import zipfile

import requests

# ──────────────────────────────────────────────────────────────────────────
# Galaxy catalogue
# distance_mpc, vflat_lit_kms: D and Vflat are columns 3 and 13 of the
#   electronic SPARC Table 1 (Lelli, McGaugh, Schombert 2016, AJ 152, 157;
#   astroweb.cwru.edu/SPARC/SPARC_Lelli2016c.mrt).
# ml_disk_default, ml_bul_default: for NGC3198/NGC6503/DDO154/NGC2403 these
#   are per-galaxy M/L values at 3.6 micron, consistent with the general trend
#   of lower M/L for bluer/gas-rich (dwarf) galaxies from stellar population
#   models (McGaugh & Schombert 2014, AJ 148, 77; Schombert & McGaugh 2014a,
#   PASA 31, 36). For NGC3198 these values (0.50/0.70) match the SPARC
#   fiducial combination Upsilon_disk=0.5, Upsilon_bul=1.4*Upsilon_disk (Lelli
#   et al. 2016, Sec. 5.1 and Fig. 5 caption). The five galaxies added later
#   (NGC5055, NGC7331, NGC2841, UGC02885, IC2574) uniformly use that same
#   SPARC fiducial 0.50/0.70 - not individually fitted from a separate
#   source, just the general SPARC default. The bulge term (Upsilon_bul) is
#   only applied to galaxies that actually have one (checked via max(Vbul) in
#   the rotmod.dat) - DDO154, NGC5055 and IC2574 have no bulge.
# ──────────────────────────────────────────────────────────────────────────
GALAXIES = {
    "NGC3198": {
        "name": "NGC 3198",
        "type": "SBc",
        "distance_mpc": 13.80,
        "ml_disk_default": 0.50,
        "ml_bul_default": 0.70,
        "vflat_lit_kms": 150.1,
        "sparc_file": "NGC3198_rotmod.dat",
        "description": "Classic flat rotation curve; no bulge; "
                        "dark matter dominates beyond ~5 kpc",
    },
    "NGC6503": {
        "name": "NGC 6503",
        "type": "Sc",
        "distance_mpc": 6.26,
        "ml_disk_default": 0.45,
        "ml_bul_default": 0.70,
        "vflat_lit_kms": 116.3,
        "sparc_file": "NGC6503_rotmod.dat",
        "description": "Isolated spiral galaxy with a clear dark-matter signature",
    },
    "DDO154": {
        "name": "DDO 154",
        "type": "IBm",
        "distance_mpc": 4.04,
        "ml_disk_default": 0.15,
        "ml_bul_default": 0.00,
        "vflat_lit_kms": 47.0,
        "sparc_file": "DDO154_rotmod.dat",
        "description": "Dwarf galaxy: baryons negligible, almost pure dark matter",
    },
    "NGC2403": {
        "name": "NGC 2403",
        "type": "SABcd",
        "distance_mpc": 3.16,
        "ml_disk_default": 0.55,
        "ml_bul_default": 0.70,
        "vflat_lit_kms": 131.2,
        "sparc_file": "NGC2403_rotmod.dat",
        "description": "Nearby spiral with extended HI measurements out to >20 kpc",
    },
    "NGC5055": {
        "name": "NGC 5055",  # Messier 63, "Sunflower Galaxy"
        "type": "Sbc",
        "distance_mpc": 9.90,
        "ml_disk_default": 0.50,
        "ml_bul_default": 0.00,
        "vflat_lit_kms": 179.0,
        "sparc_file": "NGC5055_rotmod.dat",
        "description": "Massive spiral with no bulge; one of the widest "
                        "rotation curves in this collection (Vflat~179 km/s)",
    },
    "NGC7331": {
        "name": "NGC 7331",
        "type": "Sb",
        "distance_mpc": 14.70,
        "ml_disk_default": 0.50,
        "ml_bul_default": 0.70,
        "vflat_lit_kms": 239.0,
        "sparc_file": "NGC7331_rotmod.dat",
        "description": "Massive early-type spiral with a prominent bulge; "
                        "often compared in size and shape to the Milky Way",
    },
    "NGC2841": {
        "name": "NGC 2841",
        "type": "Sb",
        "distance_mpc": 14.10,
        "ml_disk_default": 0.50,
        "ml_bul_default": 0.70,
        "vflat_lit_kms": 284.8,
        "sparc_file": "NGC2841_rotmod.dat",
        "description": "Early-type spiral with a dominant bulge (Vbul>Vdisk); "
                        "one of the highest rotation speeds in this exercise",
    },
    "UGC02885": {
        "name": "UGC 2885",  # "Rubin's Galaxy" - named after Vera Rubin
        "type": "Sc",
        "distance_mpc": 80.60,
        "ml_disk_default": 0.50,
        "ml_bul_default": 0.70,
        "vflat_lit_kms": 289.5,
        "sparc_file": "UGC02885_rotmod.dat",
        "description": "One of the largest known spiral galaxies (diameter "
                        "~2.5x the Milky Way); an extreme test of the flat "
                        "curve at large R",
    },
    "IC2574": {
        "name": "IC 2574",  # "Coddington's Nebula"
        "type": "Sm",
        "distance_mpc": 3.91,
        "ml_disk_default": 0.50,
        "ml_bul_default": 0.00,
        "vflat_lit_kms": 66.4,
        "sparc_file": "IC2574_rotmod.dat",
        "description": "Dwarf irregular galaxy with a slowly-rising curve; a "
                        "classic example of the dark-matter 'core-cusp' problem",
    },
}

# Individual *_rotmod.dat files are not available at separate URLs; SPARC
# distributes them only as one ZIP archive containing all 175 galaxies.
SPARC_ZIP_URL = "https://astroweb.cwru.edu/SPARC/Rotmod_LTG.zip"

def download_sparc(galaxy_key):
    """Download the SPARC rotmod file (from Rotmod_LTG.zip). Returns None on failure."""
    info = GALAXIES[galaxy_key]
    target_name = info["sparc_file"]
    try:
        resp = requests.get(SPARC_ZIP_URL, timeout=60)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Warning: download failed ({e})", file=sys.stderr)
        return None

    try:
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            member = next(
                (n for n in zf.namelist() if n.endswith(target_name)), None
            )
            if member is None:
                print(f"  Warning: '{target_name}' not found in archive",
                      file=sys.stderr)
                return None
            text = zf.read(member).decode("utf-8", errors="replace")
    except zipfile.BadZipFile as e:
        print(f"  Warning: download failed ({e})", file=sys.stderr)
        return None

    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) < 6:
            continue
        try:
            r, vobs, errv, vgas, vdisk, vbul = (float(p) for p in parts[:6])
            if r > 0 and vobs > 0:
                rows.append((r, vobs, errv, vgas, vdisk, vbul))
        except ValueError:
            continue
    return rows if rows else None


def compute_row(r, vobs, errv, vgas, vdisk, vbul, ml_disk, ml_bul):
    """Derived columns for one rotation-curve row."""
    vbar2 = vgas**2 + ml_disk * vdisk**2 + ml_bul * vbul**2
    vbaryon = math.sqrt(max(vbar2, 0.0))
    g_factor = 4302.0  # M[1e9 Msun] = V^2[km/s]^2 * R[kpc] / g_factor
    mobs = vobs**2 * r / g_factor
    mbar = vbar2 * r / g_factor
    mdm = max(mobs - mbar, 0.0)
    fdm = mdm / mobs if mobs > 0 else 0.0
    return vbaryon, mobs, mbar, mdm, fdm


def write_csv(rows, info, ml_disk, ml_bul, outfile):
    header_lines = [
        f"# Galaxy: {info['name']} ({info['type']})",
        f"# Distance: {info['distance_mpc']} Mpc",
        "# Source: SPARC database (Lelli, McGaugh, Schombert 2016, AJ 152, 157)",
        f"# URL:   {SPARC_ZIP_URL} (file: {info['sparc_file']})",
        f"# M/L disk: {ml_disk}  |  M/L bulge: {ml_bul}  (3.6 micron, solar units)",
        f"# Vflat (literature): {info['vflat_lit_kms']} km/s",
        f"# Note: {info['description']}",
        "#",
        "# Mass enclosed within R: M[1e9 Msun] = V^2[km/s]^2 * R[kpc] / 4302",
        "#   (G = 4.302e-3 pc Msun^-1 (km/s)^2)",
        "#",
    ]
    writer = csv.writer(outfile)
    for h in header_lines:
        outfile.write(h + "\n")
    writer.writerow([
        "R_kpc", "Vobs_kms", "errV_kms",
        "Vgas_kms", "Vdisk_kms", "Vbul_kms",
        "Vbaryon_kms",
        "Mobs_1e9Msun", "Mbar_1e9Msun", "MDM_1e9Msun", "fDM",
    ])
    for (r, vobs, errv, vgas, vdisk, vbul) in rows:
        vbaryon, mobs, mbar, mdm, fdm = compute_row(
            r, vobs, errv, vgas, vdisk, vbul, ml_disk, ml_bul
        )
        writer.writerow([
            f"{r:.3f}", f"{vobs:.1f}", f"{errv:.1f}",
            f"{vgas:.1f}", f"{vdisk:.1f}", f"{vbul:.1f}",
            f"{vbaryon:.1f}",
            f"{mobs:.3f}", f"{mbar:.3f}", f"{mdm:.3f}", f"{fdm:.3f}",
        ])


def main(galaxy, ml_disk=None, ml_bul=None, output=None):
    if galaxy not in GALAXIES:
        print(f"Unknown galaxy: {galaxy}")
        print("Available galaxies:")
        for k, v in GALAXIES.items():
            print(f"  {k:10s}  {v['name']:12s}  Vflat={v['vflat_lit_kms']:>5.0f} km/s")
        return

    info = GALAXIES[galaxy]
    ml_disk = ml_disk if ml_disk is not None and ml_disk >= 0 else info["ml_disk_default"]
    ml_bul = ml_bul if ml_bul is not None and ml_bul >= 0 else info["ml_bul_default"]
    output = output if output else f"{galaxy}.csv"

    print(f"Galaxy: {info['name']}  |  M/L disk={ml_disk}  M/L bulge={ml_bul}")
    print("  Downloading from the SPARC server...", end=" ", flush=True)
    rows = download_sparc(galaxy)
    if not rows:
        print("failed.", file=sys.stderr)
        raise RuntimeError(
            f"Could not download SPARC data for '{galaxy}' from {SPARC_ZIP_URL}. "
            "Check your internet connection and try again."
        )
    print(f"OK ({len(rows)} points)")

    with open(output, "w", newline="", encoding="utf-8") as f:
        write_csv(rows, info, ml_disk, ml_bul, f)
    print(f"Saved: {output}  ({len(rows)} rows)")

    try:
        from google.colab import files
        files.download(output)
    except ImportError:
        pass  # not running in Colab; the file stays in the working directory

In [ ]:
main(GALAXY, ML_DISK, ML_BUL, OUTPUT or None)